# 01 — Burgers' equation: FNO vs U-Net

End-to-end run: generate data, train both models, and reproduce experiments 1–3
(accuracy, resolution transfer, data efficiency).

On Colab: Runtime → Change runtime type → T4 GPU, then run all.


## Setup

In [ ]:
# On Colab, clone your repo and install deps. Locally, just run from the repo root.
# !git clone https://github.com/<your-username>/fno-pde.git
# %cd fno-pde
# !pip install -q -r requirements.txt


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from src.solvers.burgers import generate_dataset
from src.data import load_burgers, make_loaders, GaussianNormalizer
from src.models.fno import FNO1d
from src.models.unet import UNet1d
from src.metrics import relative_l2, count_params

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. Generate a small dataset
Use a modest size here so the notebook runs quickly; scale up for the real runs.

In [ ]:
import os
os.makedirs("data", exist_ok=True)
data = generate_dataset(n_samples=600, grid=256, nu=0.01, t_final=1.0, dt=1e-4, seed=0)
np.savez_compressed("data/burgers.npz", **data)
print("u0:", data["u0"].shape, " uT:", data["uT"].shape)


### Sanity check: a few input/output pairs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.plot(data["u0"][i], label="u(x,0)")
    ax.plot(data["uT"][i], label="u(x,1)")
    ax.set_title(f"sample {i}"); ax.legend()
plt.tight_layout(); plt.show()


## 2. Train the FNO

In [ ]:
(x_tr, y_tr), (x_te, y_te) = load_burgers("data/burgers.npz", grid=256, n_train=500, n_test=100)
x_norm = GaussianNormalizer(x_tr[..., 0]); y_norm = GaussianNormalizer(y_tr)
x_tr[..., 0] = x_norm.encode(x_tr[..., 0]); x_te[..., 0] = x_norm.encode(x_te[..., 0])
tr, _ = make_loaders(x_tr, y_norm.encode(y_tr), x_te, y_te, batch_size=32)

fno = FNO1d(modes=16, width=64, depth=4).to(device)
print("FNO params:", count_params(fno))
opt = torch.optim.Adam(fno.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)

for epoch in range(50):
    fno.train()
    for xb, yb in tr:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = relative_l2(fno(xb), yb); loss.backward(); opt.step()
    sched.step()
    if (epoch + 1) % 10 == 0:
        print(f"epoch {epoch+1}  train rel-L2 {loss.item():.4f}")


In [ ]:
fno.eval()
with torch.no_grad():
    pred = y_norm.decode(fno(x_te.to(device)).cpu())
print("FNO test rel-L2:", relative_l2(pred, y_te).item())


## 3. Next steps
- Train the U-Net (`configs/burgers_unet.yaml`) and compare.
- Experiment 2: train at grid=64, evaluate at 128/256 — the FNO handles it, the U-Net needs resizing.
- Experiment 3: sweep `n_train` in {100, 250, 500} and plot error vs size.
Use `python -m src.train --config ...` for the logged runs so results land in `results/metrics.csv`.